# Detective Graph Visualization

This notebook explores the extracted murder-mystery graphs in `data/graphs/`.

It does three things:
- loads every extracted graph JSON
- summarizes the dataset at a glance
- visualizes a selected graph as a network diagram


In [ ]:
%pip install networkx matplotlib pandas

In [ ]:
from __future__ import annotations

import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

PROJECT_ROOT = Path(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
GRAPHS_DIR = PROJECT_ROOT / "data" / "graphs"

if not GRAPHS_DIR.exists():
    raise FileNotFoundError(f"Could not find graphs directory: {GRAPHS_DIR}")

graph_paths = sorted(GRAPHS_DIR.glob("*.json"))
if not graph_paths:
    raise RuntimeError(f"No graph JSON files found in {GRAPHS_DIR}")

print(f"Found {len(graph_paths)} extracted graphs in {GRAPHS_DIR}")


In [ ]:
def load_graph(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def graph_record(path: Path) -> dict:
    graph = load_graph(path)
    metadata = graph.get("metadata", {})
    labels = Counter(character.get("label", "UNK") for character in graph.get("characters", []))
    return {
        "entry_id": metadata.get("entry_id", path.stem),
        "title": metadata.get("title", path.stem),
        "author": metadata.get("author", ""),
        "year": metadata.get("year", ""),
        "medium": metadata.get("medium", ""),
        "characters": len(graph.get("characters", [])),
        "occupations": len(graph.get("occupations", [])),
        "locations": len(graph.get("locations", [])),
        "organizations": len(graph.get("organizations", [])),
        "edges": len(graph.get("edges", [])),
        "villains": labels.get("Villain", 0),
        "victims": labels.get("Victim", 0),
        "witnesses": labels.get("Witness", 0),
        "uninvolved": labels.get("Uninvolved", 0),
        "unk": labels.get("UNK", 0),
    }


summary_df = pd.DataFrame(graph_record(path) for path in graph_paths).sort_values(["medium", "year", "entry_id"])
summary_df.head()


In [ ]:
numeric_summary = summary_df.select_dtypes(include="number").describe().transpose().round(2)
categorical_summary = summary_df.select_dtypes(exclude="number").describe().transpose()

print("Numeric columns")
display(numeric_summary)

print("Categorical columns")
display(categorical_summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

summary_df["characters"].plot(kind="hist", bins=20, ax=axes[0], color="#457b9d", edgecolor="white")
axes[0].set_title("Characters per graph")
axes[0].set_xlabel("Characters")

summary_df["edges"].plot(kind="hist", bins=20, ax=axes[1], color="#e76f51", edgecolor="white")
axes[1].set_title("Edges per graph")
axes[1].set_xlabel("Edges")

summary_df.groupby("medium").size().sort_values(ascending=False).plot(kind="bar", ax=axes[2], color="#2a9d8f")
axes[2].set_title("Graphs by medium")
axes[2].set_xlabel("Medium")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

label_totals = summary_df[["villains", "victims", "witnesses", "uninvolved", "unk"]].sum().sort_values(ascending=False)
ax = label_totals.plot(kind="bar", figsize=(8, 4), color=["#d62828", "#6a4c93", "#f4a261", "#8d99ae", "#adb5bd"])
ax.set_title("Character labels across all graphs")
ax.set_xlabel("Label")
ax.set_ylabel("Characters")
plt.tight_layout()
plt.show()

edge_type_totals = Counter()
for path in graph_paths:
    graph = load_graph(path)
    edge_type_totals.update(edge.get("relation", "unknown") for edge in graph.get("edges", []))

edge_type_df = pd.Series(edge_type_totals, name="count").sort_values(ascending=False)
print("Edge relation types")
display(edge_type_df.rename_axis("edge_type").reset_index())

ax = edge_type_df.head(20).plot(kind="bar", figsize=(12, 4), color="#264653")
ax.set_title("Top edge relation types across all graphs")
ax.set_xlabel("Relation")
ax.set_ylabel("Edges")
plt.tight_layout()
plt.show()


## Select a graph to visualize

Change `GRAPH_ID` below to any file stem in `data/graphs/`, for example `NOV_001` or `FLM_025`.


In [ ]:
GRAPH_ID = "NOV_001"

selected_path = GRAPHS_DIR / f"{GRAPH_ID}.json"
selected_graph = load_graph(selected_path)
selected_graph["metadata"]


In [ ]:
NODE_TYPE_STYLES = {
    "character": {"color": "#457b9d", "size": 1300},
    "occupation": {"color": "#2a9d8f", "size": 900},
    "location": {"color": "#e9c46a", "size": 900},
    "organization": {"color": "#e76f51", "size": 1000},
}

CHARACTER_LABEL_COLORS = {
    "Villain": "#d62828",
    "Victim": "#6a4c93",
    "Witness": "#f4a261",
    "Uninvolved": "#8d99ae",
    "UNK": "#adb5bd",
}


def to_visual_graph(graph: dict) -> nx.DiGraph:
    visual_graph = nx.DiGraph()

    for character in graph.get("characters", []):
        prominence = character.get("features", {}).get("narrative_prominence", 0.5)
        prominence = 0.5 if prominence == -1 else prominence
        visual_graph.add_node(
            character["id"],
            name=character["name"],
            node_type="character",
            label=character.get("label", "UNK"),
            size=NODE_TYPE_STYLES["character"]["size"] + 1200 * prominence,
        )

    for occupation in graph.get("occupations", []):
        visual_graph.add_node(
            occupation["id"],
            name=occupation["name"],
            node_type="occupation",
            label="occupation",
            size=NODE_TYPE_STYLES["occupation"]["size"],
        )

    for location in graph.get("locations", []):
        visual_graph.add_node(
            location["id"],
            name=location["name"],
            node_type="location",
            label="location",
            size=NODE_TYPE_STYLES["location"]["size"],
        )

    for organization in graph.get("organizations", []):
        visual_graph.add_node(
            organization["id"],
            name=organization["name"],
            node_type="organization",
            label="organization",
            size=NODE_TYPE_STYLES["organization"]["size"],
        )

    edge_labels = defaultdict(list)
    for edge in graph.get("edges", []):
        source = edge["source"]
        target = edge["target"]
        relation = edge.get("relation", "related")
        edge_labels[(source, target)].append(relation)

    for (source, target), relations in edge_labels.items():
        visual_graph.add_edge(source, target, relations=sorted(set(relations)))

    return visual_graph


def draw_graph(graph: dict, figsize: tuple[int, int] = (18, 12), seed: int = 7):
    visual_graph = to_visual_graph(graph)
    position = nx.spring_layout(visual_graph, seed=seed, k=1.1)

    plt.figure(figsize=figsize)
    ax = plt.gca()
    ax.set_title(graph.get("metadata", {}).get("title", "Graph visualization"), fontsize=16)
    ax.axis("off")

    for node_type, style in NODE_TYPE_STYLES.items():
        nodes = [node for node, attrs in visual_graph.nodes(data=True) if attrs["node_type"] == node_type]
        if not nodes:
            continue

        colors = []
        sizes = []
        for node in nodes:
            attrs = visual_graph.nodes[node]
            sizes.append(attrs["size"])
            if node_type == "character":
                colors.append(CHARACTER_LABEL_COLORS.get(attrs["label"], style["color"]))
            else:
                colors.append(style["color"])

        nx.draw_networkx_nodes(
            visual_graph,
            position,
            nodelist=nodes,
            node_color=colors,
            node_size=sizes,
            alpha=0.95,
            edgecolors="black",
            linewidths=0.8,
        )

    nx.draw_networkx_edges(
        visual_graph,
        position,
        edge_color="#6c757d",
        arrows=True,
        arrowsize=14,
        alpha=0.45,
        width=1.2,
        connectionstyle="arc3,rad=0.08",
    )

    display_labels = {}
    for node, attrs in visual_graph.nodes(data=True):
        if attrs["node_type"] == "character":
            display_labels[node] = f"{attrs['name']}\n[{attrs['label']}]"
        else:
            display_labels[node] = attrs["name"]

    nx.draw_networkx_labels(visual_graph, position, labels=display_labels, font_size=8)

    compact_edge_labels = {}
    for source, target, attrs in visual_graph.edges(data=True):
        relations = attrs["relations"]
        compact_edge_labels[(source, target)] = ", ".join(relations[:2])
        if len(relations) > 2:
            compact_edge_labels[(source, target)] += f" +{len(relations) - 2}"

    if visual_graph.number_of_edges() <= 40:
        nx.draw_networkx_edge_labels(
            visual_graph,
            position,
            edge_labels=compact_edge_labels,
            font_size=7,
            rotate=False,
            bbox={"alpha": 0.6, "color": "white", "pad": 0.2},
        )

    plt.show()

    return visual_graph


In [ ]:
visual_graph = draw_graph(selected_graph)
print(f"Nodes: {visual_graph.number_of_nodes()} | Edges: {visual_graph.number_of_edges()}")


In [ ]:
edge_table = []
name_lookup = {node_id: attrs["name"] for node_id, attrs in visual_graph.nodes(data=True)}
for source, target, attrs in visual_graph.edges(data=True):
    edge_table.append({
        "source": name_lookup[source],
        "target": name_lookup[target],
        "relations": ", ".join(attrs["relations"]),
    })

pd.DataFrame(edge_table).sort_values(["source", "target"]).head(25)
